# Convert raw data to 'strict' json

In [5]:
import json
import gzip
import os
dataset_name = "Beauty"
os.makedirs(dataset_name, exist_ok=True)

def parse(path):
  g = gzip.open(path, 'r')
  for l in g:
    yield json.dumps(eval(l))

# Beauty dataset
f = open(f"./{dataset_name}/{dataset_name}.json", 'w')
for l in parse(f"reviews_{dataset_name}_5.json.gz"):
  f.write(l + '\n')

In [6]:
# print the number of lines in the file and the first line
data = open(f"./{dataset_name}/{dataset_name}.json", 'r')
print("Number of lines:", sum(1 for _ in data))
data.seek(0)  # Reset file pointer to the beginning
print("First line:", data.readline().strip())
data.close()

Number of lines: 198502
First line: {"reviewerID": "A1YJEY40YUW4SE", "asin": "7806397051", "reviewerName": "Andrea", "helpful": [3, 4], "reviewText": "Very oily and creamy. Not at all what I expected... ordered this to try to highlight and contour and it just looked awful!!! Plus, took FOREVER to arrive.", "overall": 1.0, "summary": "Don't waste your money", "unixReviewTime": 1391040000, "reviewTime": "01 30, 2014"}


In [ ]:
%cd <your path>\TIGER-main\data

In [2]:
# 👈👈👈
import numpy as np
import pandas as pd
import json

# Initialize mapping dictionaries
userID_mapping = {}
itemID_mapping = {}

# Open the JSON file for reading
data = open(f"./{dataset_name}/{dataset_name}.json", 'r')

# Initialize lists to store userID, itemID, and timestamp
userIDs = []
itemIDs = []
timestamps = []

# Process each line in the JSON file
for line in data:
    review = json.loads(line.strip())
    userID = review['reviewerID']
    itemID = review['asin']
    timestamp = review['unixReviewTime']
    
    # Map userID to an integer starting from 1
    if userID not in userID_mapping:
        userID_mapping[userID] = len(userID_mapping) + 1
    
    # Map itemID to an integer starting from 1
    if itemID not in itemID_mapping:
        itemID_mapping[itemID] = len(itemID_mapping) + 1
    
    # Append mapped values and timestamp to lists
    userIDs.append(userID_mapping[userID])
    itemIDs.append(itemID_mapping[itemID])
    timestamps.append(timestamp)

# Save mapping dictionaries as .npy files
np.save(f'./{dataset_name}/user_mapping.npy', userID_mapping)
print("user_num:", len(userID_mapping))
print("the first five userID mapping:", list(userID_mapping.items())[:5])
np.save(f'./{dataset_name}/item_mapping.npy', itemID_mapping)
print("item_num:", len(itemID_mapping))
print("the first five itemID mapping:", list(itemID_mapping.items())[:5])

# Group itemIDs by userID and sort by timestamp
user_item_mapping = {}
for userID, itemID, timestamp in zip(userIDs, itemIDs, timestamps):
    if userID not in user_item_mapping:
        user_item_mapping[userID] = []
    user_item_mapping[userID].append((itemID, timestamp))

# Sort itemIDs for each user by timestamp
for userID in user_item_mapping:
    user_item_mapping[userID].sort(key=lambda x: x[1])
    user_item_mapping[userID] = [item[0] for item in user_item_mapping[userID]]

# Print a sample of the results
print("user-item mapping:", list(user_item_mapping.items())[:5])

# Split data into training, validation, and testing sets using leave-one-out strategy
train_data = {}
val_data = {}
test_data = {}

for userID, item_sequence in user_item_mapping.items():
    # Assign the last item for testing, the second-to-last for validation, and the rest for training
    train_data[userID] = item_sequence[:-2]
    val_data[userID] = item_sequence[:-1]
    test_data[userID] = item_sequence

# Print a sample of the split data
# print("training data:", list(train_data.items())[:5])
# print("validation data:", list(val_data.items())[:5])
# print("testing data:", list(test_data.items())[:5])

# Prepare data for train, validation, and test sets
def prepare_data(data_dict):
    rows = []
    for userID, item_sequence in data_dict.items():
        history = item_sequence[:-1]
        target = item_sequence[-1]
        rows.append({'user': userID, 'history': history, 'target': target})
    return pd.DataFrame(rows)

# Create dataframes for train, validation, and test sets
train_df = prepare_data(train_data)
print("\nTraining data shape:", train_df.shape)
print("the first 3 rows of training data:\n", train_df.head(3))
val_df = prepare_data(val_data)
print("\nValidation data shape:", val_df.shape)
print("the first 3 rows of validation data:\n", val_df.head(3))
test_df = prepare_data(test_data)
print("\nTesting data shape:", test_df.shape)
print("the first 3 rows of testing data:\n", test_df.head(3))

# Save dataframes to parquet files
train_df.to_parquet(f'./{dataset_name}/train.parquet', index=False)
val_df.to_parquet(f'./{dataset_name}/valid.parquet', index=False)
test_df.to_parquet(f'./{dataset_name}/test.parquet', index=False)

print("Data saved to parquet files.")

data.close()


user_num: 22363
the first five userID mapping: [('A1YJEY40YUW4SE', 1), ('A60XNB876KYML', 2), ('A3G6XNM240RMWA', 3), ('A1PQFP6SAJ6D80', 4), ('A38FVHZTNQ271F', 5)]
item_num: 12101
the first five itemID mapping: [('7806397051', 1), ('9759091062', 2), ('9788072216', 3), ('9790790961', 4), ('9790794231', 5)]
user-item mapping: [(1, [6846, 7873, 4585, 1, 5406]), (2, [816, 10406, 11194, 11651, 9716, 1, 233]), (3, [1, 6050, 7977, 5252, 4211, 243, 11204, 5863, 6609]), (4, [5522, 439, 5161, 11140, 1, 7849]), (5, [1, 10470, 10064, 9403, 10362, 4758, 6500, 11444, 11390])]

Training data shape: (22363, 3)
the first 3 rows of training data:
    user                           history  target
0     1                      [6846, 7873]    4585
1     2        [816, 10406, 11194, 11651]    9716
2     3  [1, 6050, 7977, 5252, 4211, 243]   11204

Validation data shape: (22363, 3)
the first 3 rows of validation data:
    user                                  history  target
0     1                       [684

# Generate Item Semantic Embeddings

In [5]:
# 👈👈👈
dataset_name = "Beauty"
# Open the metadata file for reading

with open(f"./{dataset_name}/{dataset_name}_metadata.json", 'r') as metadata_file:
    # Create a reverse mapping from itemID to asin
    reverse_itemID_mapping = {v: k for k, v in itemID_mapping.items()}

    # Initialize a dictionary to store the extracted information
    item_info = {}

    # Process each line in the metadata file
    for line in metadata_file:
        metadata = json.loads(line.strip())
        asin = metadata.get('asin')

        # Check if the asin exists in the reverse mapping
        if asin in reverse_itemID_mapping.values():
            itemID = itemID_mapping[asin]
            item_info[itemID] = {
                # 1. text
                # 'xxxxxxx'
                'title': metadata.get('title') if metadata.get('title') else None,
                # 190
                'price': metadata.get('price') if metadata.get('price') else None,
                # {'Beauty': 10486, "xxx": xxx}
                'salesRank': metadata.get('salesRank') if metadata.get('salesRank') else None,
                # 'COKA'
                'brand': metadata.get('brand') if metadata.get('brand') else None,
                # ['Beauty', 'Makeup', 'Face', 'Concealers & Neutralizers']
                'categories': metadata.get('categories')[0] if metadata.get('categories') else None,
                # 'xxxxxxxxxxxxxxxx'
                'description': metadata.get('description') if metadata.get('description') else None,

                # 2. image
                # 'imUrl': 'http://ecx.images-amazon.com/images/I/41Rn18OeU6L._SY300_.jpg'
                'imUrl': metadata.get('imUrl') if metadata.get('imUrl') else None,
            }
        # asin = metadata.get('asin')
        #
        # # Check if the asin exists in the reverse mapping
        # if asin in reverse_itemID_mapping.values():
        #     # for k in metadata.keys():
        #     #     print(k, ": ", metadata[k])
        #     # break

# Print the information for the first 5 items
for itemID, info in list(item_info.items())[:5]:
    print(f"ItemID: {itemID}, Info: {info}")

ItemID: 1, Info: {'title': 'WAWO 15 Color Professionl Makeup Eyeshadow Camouflage Facial Concealer Neutral Palette', 'price': 5.04, 'salesRank': {'Beauty': 10486}, 'brand': 'COKA', 'categories': ['Beauty', 'Makeup', 'Face', 'Concealers & Neutralizers'], 'description': 'An extensive range of 15 multiple vibrant long wear concealer colour with different skin tones to create more than 10,000 amazing looks. Using the most commonly applied shades, ensures the best skin colour match and guarantees a traceless and natural finish. Enabling layering and mixing, provides total camouflage for almost any skin problem including blemishes, scars, birthmarks and black circles. It is also suitable to use as bronzer. The light colour is suitable for redness, acne and so on. The medium colour is perfect for dark shadows in the under-eye area. The dark colour provides exceptional camouflage and adheres well to the skin. Silky glossy colour and high quality ingredients together to care skin around and can

In [6]:
# Prepare data for embedding
item_embeddings = []
for itemID, info in item_info.items():
    # Combine relevant fields into a single text for embedding

    item_embeddings.append({
        'ItemID': itemID,
        'title': info.get('title', ''),
        'price': info.get('price', ''),
        'salesRank': info.get('salesRank', ''),
        'brand': info.get('brand', ''),
        'categories': info.get('categories', ''),
        'image': info.get('imUrl', ''),
        'description': info.get('description', ''),
    })

# Convert to DataFrame
item_emb_df = pd.DataFrame(item_embeddings)

print("\nItem embeddings DataFrame shape:", item_emb_df.shape)
print("The first 3 rows of item embeddings DataFrame:\n", item_emb_df.head(3))


Item embeddings DataFrame shape: (12100, 8)
The first 3 rows of item embeddings DataFrame:
    ItemID                                              title  price  \
0       1  WAWO 15 Color Professionl Makeup Eyeshadow Cam...   5.04   
1       2                  Xtreme Brite Brightening Gel 1oz.  19.99   
2       3  Prada Candy By Prada Eau De Parfum Spray 1.7 O...  65.86   

           salesRank         brand  \
0  {'Beauty': 10486}          COKA   
1  {'Beauty': 52254}  Xtreme Brite   
2  {'Beauty': 78916}         Prada   

                                          categories  \
0  [Beauty, Makeup, Face, Concealers & Neutralizers]   
1  [Beauty, Hair Care, Styling Products, Creams, ...   
2        [Beauty, Fragrance, Women's, Eau de Parfum]   

                                               image  \
0  http://ecx.images-amazon.com/images/I/41Rn18Oe...   
1  http://ecx.images-amazon.com/images/I/41QWW9v1...   
2  http://ecx.images-amazon.com/images/I/51iT2k6L...   

                   

In [1]:
!git clone https://github.com/beichenzbc/Long-CLIP.git
%cd Long-CLIP

In [8]:
import pandas as pd
import torch
from transformers import CLIPProcessor, CLIPModel, CLIPConfig
from PIL import Image
import requests
from io import BytesIO
from tqdm import tqdm
import numpy as np
import os
import warnings

import transformers

# ====== 🔇 静音 transformers 输出 ======
warnings.filterwarnings("ignore", message="Token indices sequence length is longer*")
transformers.logging.set_verbosity_error()

TITLE_LIMIT = 25
CLIP_LIMIT = 77
RESERVED = 2
SAFE_LIMIT = CLIP_LIMIT - RESERVED

# ========== 1️⃣ 基础设置 ==========
dataset_name = "Beauty"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# 下载代码到./data/Long-CLIP
# 下载权重到./data/Long-CLIP/checkpoints
# 修改./data/Long-CLIP/model/longclip的 from pkg_resources import packaging 为 import packaging
# pip install ftfy
import sys
sys.path.append("./Long-CLIP")
from model import longclip
model, preprocess = longclip.load("./Long-CLIP/checkpoints/longclip-B-32.pt", device=device)


def build_limited_text(info):
    """拼接文本：
      - title ≤ 25 tokens
      - 整体 ≤ 77 tokens (含 [CLS], [EOS])
    """
    title = str(info.get("title", ""))
    text = f"title: {title}"

    fields = [
        ("categories", info.get("categories", "")),
        ("price", info.get("price", "")),
        ("salesRank", info.get("salesRank", "")),
        ("brand", info.get("brand", "")),
        ("description", info.get("description", "")),
    ]

    for k, v in fields:
        segment = f"\n{k}: {v}"
        text = text + segment
    return text.strip()
# ====================================================


# ========== 2️⃣ 辅助函数 ==========
def get_image_emb(url):
    """从URL获取图像embedding，失败则返回全0向量"""
    if not url or not isinstance(url, str) or not url.startswith("http"):
        return np.zeros(512)
    try:
        response = requests.get(url, timeout=5)
        img = Image.open(BytesIO(response.content)).convert("RGB")
        inputs = preprocess(img).unsqueeze(0).to(device)
        with torch.no_grad():
            emb = model.encode_image(inputs)
        return emb.squeeze().cpu().numpy()
    except Exception:
        print(f"no img: {url}")
        return np.zeros(512)


def get_text_emb(info):
    """获取文本embedding，使用 build_limited_text 限制 token 数"""
    text = build_limited_text(info)
    if not text or not isinstance(text, str):
        return np.zeros(512)

    inputs = longclip.tokenize([text], context_length=248, truncate=True).to(device)

    with torch.no_grad():
        emb = model.encode_text(inputs)
    return emb.squeeze().cpu().numpy()


# ========== 3️⃣ 生成embedding ==========
text_embs, img_embs = [], []

print(f"Encoding {len(item_emb_df)} items with CLIP (title≤25, total≤77)...")

for _, row in tqdm(item_emb_df.iterrows(), total=len(item_emb_df), desc="CLIP encoding"):
    text_emb = get_text_emb(row).tolist()  # 🚨 改：传入整行 row（含 title/price/...）
    img_emb = get_image_emb(row["image"]).tolist()
    text_embs.append(text_emb)
    img_embs.append(img_emb)

# ========== 4️⃣ 保存 ==========
item_emb_df_ = item_emb_df.copy()
item_emb_df_["text_emb"] = text_embs
item_emb_df_["image_emb"] = img_embs

print("✅ Embedding generation completed.")

# ========== 3️⃣ 生成embedding ==========
# text_embs, img_embs = [], []
#
# print(f"Encoding {len(item_info)} items with CLIP (title≤25, total≤77)...")
# item_embeddings = []
# for itemID, info in tqdm(item_info.items(), total=len(item_info), desc="CLIP encoding"):
#     text_emb = get_text_emb(info).tolist()  # 🚨 改：传入整行 row（含 title/price/...）
#     img_emb = get_image_emb(info["imUrl"]).tolist()
#
#     text_embs.append(text_emb)
#     img_embs.append(img_emb)
#
#     item_embeddings.append({'ItemID': itemID, 'text_emb': text_embs, 'img_emb': img_embs})
#
# # ========== 4️⃣ 保存 ==========
# item_emb_df = pd.DataFrame(item_embeddings)
#
# print("\nItem embeddings DataFrame shape:", item_emb_df.shape)
# print("The first 3 rows of item embeddings DataFrame:\n", item_emb_df.head(3))

Using device: cuda
Encoding 12100 items with CLIP (title≤25, total≤77)...


CLIP encoding: 100%|██████████| 12100/12100 [13:16<00:00, 15.20it/s]

✅ Embedding generation completed.


In [9]:
item_emb_df_.to_parquet(f'./{dataset_name}/item_longclip_emb_t_i.parquet', index=False)

In [13]:
# item_emb_df_2 = item_emb_df_.copy()
# item_emb_df_2["embedding"] = item_emb_df_2.apply(
#     lambda row: np.concatenate([np.array(row["text_emb"]), np.array(row["image_emb"])]).tolist(),
#     axis=1
# )
SEP_DIM = 1
SEP_VALUE = 0.0  # 或者 1.0
sep = np.full((SEP_DIM,), SEP_VALUE, dtype=np.float32)

item_emb_df_2 = item_emb_df_.copy()
item_emb_df_2["embedding"] = item_emb_df_2.apply(
    lambda row: np.concatenate([
        np.asarray(row["text_emb"],  dtype=np.float32),
        sep,
        np.asarray(row["image_emb"], dtype=np.float32)
    ], axis=0).tolist(),
    axis=1
)

In [14]:
item_emb_df_2.head(5)

,ItemID,title,price,salesRank,brand,categories,image,description,text_emb,image_emb,embedding
0,1,WAWO 15 Color Professionl Makeup Eyeshadow Cam...,5.04,{'Beauty': 10486},COKA,"[Beauty, Makeup, Face, Concealers & Neutralizers]",http://ecx.images-amazon.com/images/I/41Rn18Oe...,An extensive range of 15 multiple vibrant long...,"[-0.283935546875, 0.0015316009521484375, 0.258...","[-0.36474609375, 0.306396484375, 0.69970703125...","[-0.283935546875, 0.0015316009521484375, 0.258..."
1,2,Xtreme Brite Brightening Gel 1oz.,19.99,{'Beauty': 52254},Xtreme Brite,"[Beauty, Hair Care, Styling Products, Creams, ...",http://ecx.images-amazon.com/images/I/41QWW9v1...,Xtreme Brite Brightening gel is a highly conc...,"[-0.0887451171875, 0.10931396484375, -0.070434...","[0.09649658203125, 0.1422119140625, -0.1755371...","[-0.0887451171875, 0.10931396484375, -0.070434..."
2,3,Prada Candy By Prada Eau De Parfum Spray 1.7 O...,65.86,{'Beauty': 78916},Prada,"[Beauty, Fragrance, Women's, Eau de Parfum]",http://ecx.images-amazon.com/images/I/51iT2k6L...,Prada Candy By Prada Eau De Parfum Spray 1.7 O...,"[-0.49658203125, -0.38720703125, 0.00848388671...","[-0.328369140625, -0.49072265625, -0.172607421...","[-0.49658203125, -0.38720703125, 0.00848388671..."
3,4,Versace Bright Crystal Eau de Toilette Spray f...,52.33,{'Beauty': 764},Versace,"[Beauty, Fragrance, Women's, Eau de Toilette]",http://ecx.images-amazon.com/images/I/418LYGLE...,Versace Bright Crystal Perfume for Women 3 oz ...,"[-0.2059326171875, -0.17626953125, -0.14575195...","[-0.273193359375, -0.28125, 0.294189453125, -0...","[-0.2059326171875, -0.17626953125, -0.14575195..."
4,5,Stella McCartney Stella,NaN,{'Beauty': 142503},None,"[Beauty, Fragrance, Women's, Eau de Parfum]",http://ecx.images-amazon.com/images/I/31L2n60J...,STELLA For Women By STELLA MCCARTNEY 1.7 oz ED...,"[-0.128662109375, -0.16796875, 0.19384765625, ...","[-0.3623046875, 0.160400390625, 0.27685546875,...","[-0.128662109375, -0.16796875, 0.19384765625, ..."


In [12]:
# Save to parquet file
item_emb_df_2.to_parquet(f'./{dataset_name}/item_longclip_emb.parquet', index=False)

print("Item embeddings saved to item_emb.parquet.")

Item embeddings saved to item_emb.parquet.


In [16]:
# 转成 numpy 数组
text_emb = np.stack(item_emb_df['text_emb'].values)
image_emb = np.stack(item_emb_df['image_emb'].values)

# 分别保存
np.save(f'./{dataset_name}/item_text_emb.npy', text_emb)
np.save(f'./{dataset_name}/item_image_emb.npy', image_emb)

print(f"✅ Saved item_text_emb.npy and item_image_emb.npy to ./{dataset_name}/")
print("text_emb shape:", text_emb.shape)
print("image_emb shape:", image_emb.shape)

✅ Saved item_text_emb.npy and item_image_emb.npy to ./Beauty/
text_emb shape: (12101, 512)
image_emb shape: (12101, 512)
